# 05 — Survival Analysis

Declarative survival analysis workflow using reusable RetainAI survival modules.

In [1]:
from __future__ import annotations

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

%load_ext autoreload
%autoreload 2

In [2]:
from modules.io.storage import load_dataframe

from modules.survival.preprocessing import (
    prepare_survival_dataset,
    prepare_cox_features,
)
from modules.survival.kaplan_meier import (
    fit_kaplan_meier,
    fit_kaplan_meier_by_group,
    survival_probabilities,
)
from modules.survival.coxph import (
    fit_cox_ph,
    hazard_ratio_summary,
)
from modules.survival.metrics import cox_concordance_index
from modules.survival.visualization import (
    save_kaplan_meier_plot,
    save_group_kaplan_meier_plot,
    save_hazard_ratio_plot,
)
from modules.survival.report import generate_survival_report

from retainai.core.paths import PROJECT_ROOT

In [3]:
processed_path = PROJECT_ROOT / "data/processed/ibm_hr_attrition_processed.csv"

df = load_dataframe(processed_path)
df.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


In [4]:
drop_columns = [
    "EmployeeNumber",
    "EmployeeCount",
    "Over18",
    "StandardHours",
]

survival_df = prepare_survival_dataset(
    df,
    duration_col="YearsAtCompany",
    event_col="Attrition",
    positive_class="Yes",
    drop_columns=drop_columns,
)

survival_df.head()

,Age,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EnvironmentSatisfaction,Gender,HourlyRate,...,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,duration,event
0,41,Travel_Rarely,1102,Sales,1,2,Life Sciences,2,Female,94,...,0,8,0,1,6,4,0,5,6.00,1
1,49,Travel_Frequently,279,Research & Development,8,1,Life Sciences,3,Male,61,...,1,10,3,3,10,7,1,7,10.00,0
2,37,Travel_Rarely,1373,Research & Development,2,2,Other,4,Male,92,...,0,7,3,3,0,0,0,0,0.01,1
3,33,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,4,Female,56,...,0,8,3,3,8,7,3,0,8.00,0
4,27,Travel_Rarely,591,Research & Development,2,1,Medical,1,Male,40,...,1,6,3,3,2,2,2,2,2.00,0


In [5]:
kmf = fit_kaplan_meier(survival_df)

kmf.survival_function_.head()

,Overall Survival
timeline,
0.00,1.000000
0.01,0.989116
1.00,0.948192
2.00,0.927792
3.00,0.911342


In [6]:
survival_probabilities(
    kmf,
    times=[1, 2, 3, 5, 10],
)

,time,survival_probability
0,1,0.948192
1,2,0.927792
2,3,0.911342
3,5,0.872931
4,10,0.776815


In [ ]:
save_kaplan_meier_plot(
    kmf,
    PROJECT_ROOT / "artifacts/figures/survival/kaplan_meier_overall.png",
)

In [ ]:
overtime_models = fit_kaplan_meier_by_group(
    survival_df,
    group_col="OverTime",
)

save_group_kaplan_meier_plot(
    overtime_models,
    PROJECT_ROOT / "artifacts/figures/survival/kaplan_meier_by_overtime.png",
    title="Kaplan-Meier Survival by OverTime",
)

In [ ]:
department_models = fit_kaplan_meier_by_group(
    survival_df,
    group_col="Department",
)

save_group_kaplan_meier_plot(
    department_models,
    PROJECT_ROOT / "artifacts/figures/survival/kaplan_meier_by_department.png",
    title="Kaplan-Meier Survival by Department",
)

In [ ]:
jobrole_models = fit_kaplan_meier_by_group(
    survival_df,
    group_col="JobRole",
)

save_group_kaplan_meier_plot(
    jobrole_models,
    PROJECT_ROOT / "artifacts/figures/survival/kaplan_meier_by_jobrole.png",
    title="Kaplan-Meier Survival by JobRole",
)

In [11]:
cox_df = prepare_cox_features(survival_df)

cox_model = fit_cox_ph(cox_df)

cox_model.summary.head()

,coef,exp(coef),se(coef),coef lower 95%,coef upper 95%,exp(coef) lower 95%,exp(coef) upper 95%,cmp to,z,p,-log2(p)
covariate,,,,,,,,,,,
Age,-0.019131,0.981051,0.006558,-0.031984,-0.006278,0.968522,0.993742,0.0,-2.917283,0.003531,8.145729
DailyRate,-0.000155,0.999845,0.000128,-0.000406,0.000095,0.999594,1.000095,0.0,-1.217921,0.223254,2.163243
DistanceFromHome,0.014442,1.014546,0.006233,0.002225,0.026658,1.002227,1.027017,0.0,2.316931,0.020508,5.607704
Education,-0.004713,0.995298,0.050157,-0.103020,0.093593,0.902109,1.098113,0.0,-0.093973,0.925130,0.112272
EnvironmentSatisfaction,-0.147085,0.863221,0.046709,-0.238633,-0.055537,0.787704,0.945977,0.0,-3.148964,0.001639,9.253405


In [12]:
hazard_summary = hazard_ratio_summary(cox_model)

hazard_summary.head(15)

,feature,coef,exp(coef),p,coef lower 95%,coef upper 95%
43,OverTime_Yes,0.805381,2.237548,9.726252e-14,0.593345,1.017417
40,JobRole_Sales Representative,0.643560,1.903244,3.407360e-03,0.212825,1.074295
34,JobRole_Laboratory Technician,0.405126,1.499491,6.439005e-03,0.113689,0.696563
23,BusinessTravel_Travel_Frequently,0.391966,1.479888,6.612347e-03,0.109084,0.674848
42,MaritalStatus_Single,0.361498,1.435478,4.533643e-03,0.111880,0.611116
31,EducationField_Technical Degree,0.254245,1.289488,1.540494e-01,-0.095355,0.603845
33,JobRole_Human Resources,0.202666,1.224664,4.642898e-01,-0.340129,0.745462
28,EducationField_Marketing,0.170950,1.186432,3.396057e-01,-0.179914,0.521814
32,Gender_Male,0.138594,1.148658,1.912309e-01,-0.069249,0.346437
26,Department_Sales,0.124824,1.132949,4.116119e-01,-0.173145,0.422794


In [13]:
partial_hazard = cox_model.predict_partial_hazard(cox_df)

cindex = cox_concordance_index(
    cox_df,
    partial_hazard,
)

cindex

0.9508913928068013

In [ ]:
save_hazard_ratio_plot(
    hazard_summary,
    PROJECT_ROOT / "artifacts/figures/survival/hazard_ratios.png",
)

In [ ]:
generate_survival_report(
    survival_probabilities=survival_probabilities(
        kmf,
        times=[1, 2, 3, 5, 10],
    ),
    hazard_summary=hazard_summary,
    concordance_index=cindex,
    output_path=PROJECT_ROOT / "artifacts/reports/survival_report.md",
)